<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day06_practice3_WnB_%EC%8B%A4%ED%97%98%EC%B6%94%EC%A0%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U wandb

In [ ]:
import os
os.environ["WANDB_MODE"] = "online"
import wandb
print("WANDB_MODE =", os.environ.get("WANDB_MODE"))

WANDB_MODE = online


In [ ]:
wandb.login() # 로그인 (방법 A: 직접 입력)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: deartoms44 (deartoms44-kwu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# 로그인 (방법 B: Colab Secrets - 권장 방법)
import os
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
import wandb
wandb.login()

True

In [ ]:
# 피마 인디언 당뇨 예측
# "CSV → 판다스 탐색 → 텐서 → Dataset/DataLoader → 학습" 전체 파이프라인

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 셀 1. 재현성 - 시드 고정은 실험 기록의 전체


In [ ]:
# 셀 2. 데이터 - 피마 로드
CSV_URL  = "https://raw.githubusercontent.com/taehojo/deeplearning_4th/master/data/pima-indians-diabetes3.csv"
CSV_PATH = "pima-indians-diabetes3.csv"

def load_pima():
  import os
  if os.path.exists(CSV_PATH):
    print(f"로컬 파일 재사용: {CSV_PATH}")
    return pd.read_csv(CSV_PATH)

  try:
    df = pd.read_csv(CSV_URL)
    df.to_csv(CSV_PATH, index = False)
    print(f"다운로드 완료 → {CSV_PATH} 저장 (다음 실행부턴 재사용)")
    return df
  except Exception as e:
    raise SystemExit(
        f"데이터 다운로드 실패: {e}\n"
    )

df = load_pima()
print("데이터 크기:", df.shape)
print(df.head())

로컬 파일 재사용: pima-indians-diabetes3.csv
데이터 크기: (768, 9)
   pregnant  plasma  pressure  thickness  insulin   bmi  pedigree  age  \
0         6     148        72         35        0  33.6     0.627   50   
1         1      85        66         29        0  26.6     0.351   31   
2         8     183        64          0        0  23.3     0.672   32   
3         1      89        66         23       94  28.1     0.167   21   
4         0     137        40         35      168  43.1     2.288   33   

   diabetes  
0         1  
1         0  
2         1  
3         0  
4         1  


In [ ]:
X, y = df.iloc[:, :-1].values, df.iloc[:, -1].values

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.4,
    random_state=42, stratify=y
)
X_val, X_te, y_val, _ = train_test_split(
    X_tmp, y_tmp, test_size=0.5,
    random_state=42, stratify=y_tmp
)

scaler = StandardScaler()
def to_t(Xa, ya, fit=False):
  Xs = scaler.fit_transform(Xa) if fit else scaler.transform(Xa)
  return (torch.tensor(Xs, dtype=torch.float32).to(device),
          torch.tensor(ya, dtype=torch.float32).reshape(-1, 1).to(device))
X_tr_t, y_tr_t = to_t(X_tr, y_tr, fit=True)
X_val_t, y_val_t = to_t(X_val, y_val)


print(f"학습 {len(X_tr)} / 검증 {len(X_val)}")

학습 460 / 검증 154


In [ ]:
# 셀 3. 데이터로더 + 모델 정의
from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(    # 학습 데이터를 배치 단위로 공급
    TensorDataset(X_tr_t.cpu(), y_tr_t.cpu()),
    batch_size=32, shuffle=True,
    generator=torch.Generator().manual_seed(42)) # 셔플 난수도 시드 고정

def make_model():
  return nn.Sequential(
      nn.Linear(8, 64), nn.ReLU(),
      nn.Linear(64, 32), nn.ReLU(),
      nn.Linear(32, 1), nn.Sigmoid())

In [ ]:
# 셀 4. WSB 패턴 - init(시작) -> log(기록) -> finish(마침) (학습률 3종 비교)
def train_with_wandb(lr, epochs=40):
  run = wandb.init(           #1. init: 실험 하나(run) 시작
      project="dl-day06-learning-rate",
      name=f"lr_{lr}",
      config={"lr": lr, "epochs":epochs, "batch_size": 32, "seed": 42},
      reinit="finish_previous", # 이전 run 이 안 닫혔으면 자동으로 닫고 새로 시작
  )

  torch.manual_seed(42) #lr 만 바꾸고 시드는 고정(공정 비교)
  model = make_model().to(device)
  loss_fn = nn.BCELoss()
  opt = torch.optim.Adam(model.parameters(), lr=run.config.lr)

  for epoch in range(epochs):
    model.train()
    for xb, yb in train_loader:
      xb, yb = xb.to(device),yb.to(device)
      loss = loss_fn(model(xb), yb)
      opt.zero_grad(); loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
      val_loss = loss_fn(model(X_val_t), y_val_t).item()
      val_acc = ((model(X_val_t) > 0.5) == y_val_t.bool()).float().mean().item()

    wandb.log({"val_loss": val_loss, "val_acc": val_acc}) # 2. log: 매 애폭 지표 기록 -> 웹 차트
  wandb.finish() # 3. finish: 이 run 을 닫음 (다음 run 과 안 엉키게)
  return val_loss, val_acc
for lr in [0.05, 0.005, 0.0005]: # 학습률 3종 비교
  val_loss, val_acc = train_with_wandb(lr)
  print(f"[run] lr={lr:<7} 최종 val loss {val_loss:.3f} | val acc {val_acc:.3f}")

val_acc,▆▆▆▂▂▄▄▄▃▃▅▄▅█▇▄▁▇▄▆▇▃▄▄▄▅▂▃▂▄▃▄▄▃▄▄▃▄▄▅
val_loss,▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▁▂▂▁▄▄▄▄▆▄▆▄▄▁▄▂▄▇████▆██
val_acc,0.75974
val_loss,2.64936


[run] lr=0.05    최종 val loss 2.649 | val acc 0.760


val_acc,▁▄▆▂▆▄▃▆▇▇▅▅██▄▇▆▄█▇▇▆▆▄█▆▆▄▂▂▇▆▇▃▃▂▃▃▃▃
val_loss,▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▃▂▃▂▄▃▄▄▆▄▅▄▆█▆▅▆█▆▆
val_acc,0.75325
val_loss,0.67939


[run] lr=0.005   최종 val loss 0.679 | val acc 0.753


val_acc,▁▁▁▂▃▅▆▆▆▅▅▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██▇▇█▇█
val_loss,█▇▆▆▅▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,0.77273
val_loss,0.43886


[run] lr=0.0005  최종 val loss 0.439 | val acc 0.773
